In [0]:
 %pip install deltalake

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
def get_squad3_client():
    return get_adls_client().get_file_system_client("squad3")
 
def ler_delta_silver_squad3(nome_tabela: str) -> "pyspark.sql.DataFrame":
    container_client = get_squad3_client()
    path_silver       = f"silver/{nome_tabela}"
    paths = [
        p.name
        for p in container_client.get_paths(path=path_silver, recursive=True)
        if p.name.endswith(".parquet") and "_delta_log" not in p.name
    ]
    dfs = []
    for file_path in paths:
        file_client = container_client.get_file_client(file_path)
        data        = file_client.download_file().readall()
        dfs.append(pd.read_parquet(io.BytesIO(data)))
    return spark.createDataFrame(pd.concat(dfs, ignore_index=True))
 
df_raw = ler_delta_silver_squad3("ecommerce_pedidos")
log.info(f"Total bruto lido: {df_raw.count():,}")

In [0]:

# DBTITLE 1,Funcao de leitura squad3 (mesma do notebook anterior)
def get_squad3_client():
    return get_adls_client().get_file_system_client("squad3")
 
def ler_delta_silver_squad3(nome_tabela: str) -> "pyspark.sql.DataFrame":
    container_client = get_squad3_client()
    path_silver       = f"silver/{nome_tabela}"
    paths = [
        p.name
        for p in container_client.get_paths(path=path_silver, recursive=True)
        if p.name.endswith(".parquet") and "_delta_log" not in p.name
    ]
    dfs = []
    for file_path in paths:
        file_client = container_client.get_file_client(file_path)
        data        = file_client.download_file().readall()
        dfs.append(pd.read_parquet(io.BytesIO(data)))
    return spark.createDataFrame(pd.concat(dfs, ignore_index=True))
 
df_raw = ler_delta_silver_squad3("ecommerce_pedidos")
log.info(f"Total bruto lido: {df_raw.count():,}")
 
# COMMAND ----------
 
# DBTITLE 1,PASSO 1 - As duplicatas sao identicas ou tem status diferentes?
from pyspark.sql.functions import col, count as spark_count
 
# Pega um id_pedido que se repete e mostra todas as suas linhas lado a lado
id_exemplo = df_raw.groupBy("id_pedido") \
    .agg(spark_count("*").alias("qtd")) \
    .filter(col("qtd") > 1) \
    .limit(1) \
    .collect()[0]["id_pedido"]
 
log.info(f"Inspecionando id_pedido de exemplo: {id_exemplo}")
 
df_raw.filter(col("id_pedido") == id_exemplo) \
    .select("id_pedido", "status_pedido", "dt_pedido", "dt_ultima_atualizacao_status", "silver_processed_at") \
    .show(truncate=False)
 
# COMMAND ----------
 
# DBTITLE 1,PASSO 1b - Confirmacao em escala: quantos ids tem status DIFERENTE entre copias?
from pyspark.sql.functions import countDistinct
 
df_variacao_status = df_raw.groupBy("id_pedido") \
    .agg(countDistinct("status_pedido").alias("qtd_status_distintos"))
 
ids_com_variacao = df_variacao_status.filter(col("qtd_status_distintos") > 1).count()
total_ids = df_variacao_status.count()
 
log.info(f"Total de id_pedido               : {total_ids:,}")
log.info(f"id_pedido com status QUE MUDA entre copias : {ids_com_variacao:,}")
log.info(f"Percentual                        : {round(100*ids_com_variacao/total_ids, 2)}%")
 
if ids_com_variacao / total_ids < 0.02:
    log.info("CONCLUSAO: duplicatas sao essencialmente identicas (reprocessamento). "
              "Dedup simples por id_pedido (mantendo qualquer copia) e seguro.")
else:
    log.warning("CONCLUSAO: existe variacao real de status entre copias. "
                "Dedup precisa manter a copia mais RECENTE (maior "
                "dt_ultima_atualizacao_status ou silver_processed_at), nao qualquer copia.")
 
# COMMAND ----------
 
# DBTITLE 1,PASSO 2 - Dedup correto (mantendo sempre a copia mais recente, por seguranca)
from pyspark.sql import Window
from pyspark.sql.functions import row_number, upper, trim
 
w = Window.partitionBy("id_pedido").orderBy(
    col("dt_ultima_atualizacao_status").desc(),
    col("silver_processed_at").desc()
)
 
df_pedidos = df_raw \
    .withColumn("_rn", row_number().over(w)) \
    .filter(col("_rn") == 1) \
    .drop("_rn") \
    .withColumn("status_norm", upper(trim(col("status_pedido"))))
 
total_dedup = df_pedidos.count()
log.info(f"Total apos dedup: {total_dedup:,} (esperado: 90.335)")
 
# COMMAND ----------
 
# DBTITLE 1,PASSO 3 - Distribuicao real de status (pos-dedup)
df_status_final = df_pedidos.groupBy("status_norm") \
    .agg(spark_count("*").alias("qtd")) \
    .orderBy(col("qtd").desc())
 
df_status_final.show(truncate=False)
 
for row in df_status_final.collect():
    pct = round(100 * row["qtd"] / total_dedup, 2)
    log.info(f"  {row['status_norm']:<20} {row['qtd']:>8,}  ({pct}%)")
 
# COMMAND ----------
 
# DBTITLE 1,PASSO 4 - Janela de maturacao (agora com dados reais, varios anos)
from pyspark.sql.functions import datediff, current_timestamp, percentile_approx, min as spark_min, max as spark_max, round as spark_round
 
df_pedidos = df_pedidos.withColumn(
    "idade_dias", datediff(current_timestamp(), col("dt_pedido"))
)
 
df_idade_status = df_pedidos.groupBy("status_norm").agg(
    spark_count("*").alias("qtd"),
    spark_min("idade_dias").alias("idade_min"),
    spark_round(percentile_approx("idade_dias", 0.5), 1).alias("idade_mediana"),
    spark_round(percentile_approx("idade_dias", 0.9), 1).alias("idade_p90"),
    spark_max("idade_dias").alias("idade_max"),
).orderBy(col("qtd").desc())
 
df_idade_status.show(truncate=False)
 
# COMMAND ----------
 
# DBTITLE 1,PASSO 5 - scale_pos_weight final (baseado no dado deduplicado e correto)
positivos = df_pedidos.filter(col("status_norm") == "CANCELADO").count()
negativos = total_dedup - positivos
scale_pos_weight = round(negativos / positivos, 4)
 
log.info(f"Positivos (Cancelado) : {positivos:,}  ({round(100*positivos/total_dedup,2)}%)")
log.info(f"Negativos             : {negativos:,}  ({round(100*negativos/total_dedup,2)}%)")
log.info(f"scale_pos_weight      : {scale_pos_weight}")
 